# 🚀 SigLIP-384 + DINOv2 Visual Search — Tuần 4
## Hệ thống tìm kiếm 2 bước: SigLIP (retrieval) → DINOv2 (rerank)
### 👤 Mã Gia Vỹ | Nhóm 3 | Lead Developer

---

| Hạng mục | Tuần 3 (Baseline) | Tuần 4 (Mục tiêu) |
|---|---|---|
| **Backbone chính** | ResNet50 (2048-dim) | **SigLIP so400m-patch14-384 (1152-dim)** |
| **Reranker** | pHash only | **DINOv2 vitb14 (768-dim) + pHash** |
| **Đặc trưng văn bản** | TF-IDF raw | **TF-IDF char_wb (3,5) + SVD (256-dim)** |
| **Fusion** | Weighted Late Fusion | **Grid Search α + Late Fusion** |
| **Query Expansion** | Không | **AQE (Average Query Expansion)** |
| **mAP@5** | 0.7635 | **>= 0.80** |

---

### 🧠 Chiến lược "Best of Both Worlds" (theo CLAUDE.md):
- **Bước 1 — Retrieval:** SigLIP so400m-patch14-384 trích xuất semantic embeddings 1152-dim, fusion với TF-IDF, dùng FAISS lấy top-100 ứng viên
- **Bước 2 — Rerank:** DINOv2 vitb14 tính lại điểm cosine similarity thuần thị giác cho top-100, kết hợp với điểm SigLIP để rerank
- **pHash Boost:** Boost cho ảnh có perceptual hash gần nhau
- **AQE:** Average Query Expansion mở rộng vector query bằng trung bình top-3 kết quả

> ⚠️ **NGHIÊM CẤM DATA LEAKAGE:** Chỉ tune siêu tham số trên Validation Set. Test Set chỉ dùng để báo cáo kết quả CUỐI CÙNG.

### 📋 Quy trình Notebook:
1. **Bước 1:** Phân chia Validation/Test Set (20%/80%) — **KHÔNG DATA LEAKAGE**
2. **Bước 2:** Trích xuất SigLIP features (1152-dim) — backbone chính
3. **Bước 3:** Trích xuất DINOv2 features (768-dim) — dùng cho rerank
4. **Bước 4:** Trích xuất TF-IDF features (256-dim sau SVD)
5. **Bước 5:** Grid Search α trên **Validation Set**
6. **Bước 6:** Đánh giá cuối + AQE + DINOv2 Reranking + pHash Boost trên **Test Set**


In [ ]:
# ============================================================
# 📦 CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT (Google Colab)
# ============================================================

# Cài đặt FAISS (thử GPU trước, nếu không có thì dùng CPU)
try:
    import faiss
    print('✅ FAISS đã được cài đặt sẵn.')
except ImportError:
    print('🔄 Đang cài đặt faiss-gpu...')
    import subprocess
    result = subprocess.run(
        ['pip', 'install', 'faiss-gpu', '-q'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('⚠️  faiss-gpu thất bại, cài faiss-cpu...')
        subprocess.run(['pip', 'install', 'faiss-cpu', '-q'])
    print('✅ Đã cài xong FAISS!')

# Cài đặt imagehash để tính pHash
try:
    import imagehash
    print('✅ imagehash đã được cài đặt sẵn.')
except ImportError:
    print('🔄 Đang cài đặt imagehash...')
    import subprocess
    subprocess.run(['pip', 'install', 'imagehash', '-q'])
    print('✅ Đã cài xong imagehash!')

# Cài đặt Hugging Face transformers nếu chưa có
try:
    import transformers
    print('✅ transformers đã được cài đặt sẵn.')
except ImportError:
    print('🔄 Đang cài đặt transformers...')
    import subprocess
    subprocess.run(['pip', 'install', 'transformers', '-q'])
    print('✅ Đã cài xong transformers!')

# Cài đặt timm (PyTorch Image Models) để dùng DINOv2
try:
    import timm
    print('✅ timm đã được cài đặt sẵn.')
except ImportError:
    print('🔄 Đang cài đặt timm...')
    import subprocess
    subprocess.run(['pip', 'install', 'timm', '-q'])
    print('✅ Đã cài xong timm!')

print('\n🎉 Tất cả thư viện đã sẵn sàng!')


In [ ]:
# ============================================================
# 🔗 KẾT NỐI GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Đã kết nối Google Drive thành công!')

In [ ]:
# ============================================================
# 📚 IMPORT CÁC THƯ VIỆN
# ============================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Hugging Face transformers (dùng SigLIP)
from transformers import SiglipVisionModel, SiglipImageProcessor

# FAISS
import faiss

# pHash
import imagehash

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split

# Tiến trình
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print('🖥️  THÔNG TIN PHẦN CỨNG')
print('=' * 60)
print(f'Thiết bị PyTorch : {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Tên GPU          : {gpu_name}')
    print(f'VRAM             : {gpu_mem:.2f} GB')
    print(f'CUDA Version     : {torch.version.cuda}')
else:
    print('⚠️  KHÔNG CÓ GPU — Tốc độ xử lý sẽ rất chậm!')
print('=' * 60)

In [ ]:
# ============================================================
# ⚙️  CẤU HÌNH DỰ ÁN — THAY ĐỔI ĐƯỜNG DẪN TẠI ĐÂY
# ============================================================

# --- Đường dẫn gốc trên Google Drive ---
BASE_DIR        = '/content/drive/MyDrive/DuLieuPython'

# --- Đường dẫn thư mục ảnh (chứa 34,250 ảnh Shopee) ---
IMAGE_DIR       = os.path.join(BASE_DIR, 'train_images')

# --- File CSV chính (34,250 dòng) ---
CANDIDATE_CSV   = os.path.join(BASE_DIR, 'train.csv')

# --- Thư mục lưu features đã trích xuất ---
PROCESSED_DIR  = os.path.join(BASE_DIR, 'processed_siglip_dino')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Thư mục lưu kết quả ---
RESULTS_DIR    = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Cấu hình mô hình SigLIP (backbone chính) ---
# Dùng phiên bản 384 lớn hơn để có embedding chất lượng cao nhất
SIGLIP_MODEL_NAME = 'google/siglip-so400m-patch14-384'

# --- Cấu hình mô hình DINOv2 (backbone reranker) ---
# DINOv2 ViT-B/14: 768-dim, xuất sắc về chi tiết thị giác
DINOV2_MODEL_NAME = 'dinov2_vitb14'

# --- Siêu tham số ---
BATCH_SIZE_SIGLIP = 16    # Batch size SigLIP-384 (model lớn, cần VRAM nhiều)
BATCH_SIZE_DINO   = 32    # Batch size DINOv2 (nhẹ hơn SigLIP-384)
NUM_WORKERS    = 2        # Số luồng DataLoader
IMG_SIZE_SIGLIP = 384     # Kích thước ảnh SigLIP so400m-patch14-384
IMG_SIZE_DINO   = 224     # Kích thước ảnh DINOv2
TOP_K          = 5        # Số kết quả cuối cùng trả về
TOP_RERANK     = 100      # Số ứng viên từ SigLIP trước khi rerank bằng DINOv2
SVD_DIM        = 256      # Số chiều TF-IDF sau SVD

# --- Tham số pHash Boosting ---
PHASH_THRESHOLD = 5       # Boost khi Hamming distance <= 5
PHASH_BOOST     = 0.15    # Boost nhẹ để không lấn át fusion score

# --- Tham số DINOv2 Reranking ---
# Điểm cuối = w_siglip * score_siglip + w_dino * score_dino
W_SIGLIP = 0.5   # Trọng số SigLIP trong rerank score (sẽ grid search)
W_DINO   = 0.5   # Trọng số DINOv2 trong rerank score (1 - W_SIGLIP)

# --- Tham số AQE ---
AQE_K    = 3     # Số top results dùng để mở rộng query

print('✅ Cấu hình đã sẵn sàng!')
print(f'   BASE_DIR         : {BASE_DIR}')
print(f'   IMAGE_DIR        : {IMAGE_DIR}')
print(f'   CANDIDATE_CSV    : {CANDIDATE_CSV}')
print(f'   PROCESSED_DIR    : {PROCESSED_DIR}')
print(f'   RESULTS_DIR      : {RESULTS_DIR}')
print(f'   SigLIP Model     : {SIGLIP_MODEL_NAME}')
print(f'   DINOv2 Model     : {DINOV2_MODEL_NAME}')
print(f'   Batch SigLIP     : {BATCH_SIZE_SIGLIP}  |  IMG_SIZE: {IMG_SIZE_SIGLIP}')
print(f'   Batch DINOv2     : {BATCH_SIZE_DINO}  |  IMG_SIZE: {IMG_SIZE_DINO}')
print(f'   TOP_K            : {TOP_K}  |  TOP_RERANK: {TOP_RERANK}')
print(f'   PHASH_THRES      : {PHASH_THRESHOLD}  |  PHASH_BOOST: {PHASH_BOOST}')


In [ ]:
# ============================================================
# 🛠️  CÁC HÀM TIỆN ÍCH (HELPER FUNCTIONS)
# ============================================================

def l2_normalize(features: np.ndarray) -> np.ndarray:
    """
    Chuẩn hóa L2 cho ma trận features.
    Mỗi hàng sẽ có norm = 1 sau khi chuẩn hóa.
    """
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    # Tránh chia cho 0
    norms = np.where(norms < 1e-10, 1e-10, norms)
    return (features / norms).astype(np.float32)


def clean_text(text: str) -> str:
    """
    Làm sạch văn bản tiêu đề sản phẩm:
    - Chuyển về chữ thường
    - Giữ lại chữ cái (bao gồm Unicode), chữ số và khoảng trắng
    - Loại bỏ khoảng trắng thừa
    """
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    # Giữ lại ký tự Latin, Unicode (tiếng Việt/Thái/v.v.), chữ số
    text = re.sub(r'[^\w\s\u00C0-\u024F\u0E00-\u0E7F\u1E00-\u1EFF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def hex_to_phash_array(hex_str) -> np.ndarray:
    """
    Chuyển chuỗi hex của pHash thành mảng bool (64 bits).
    Dùng cho tính khoảng cách Hamming vectorized.
    """
    try:
        phash_obj = imagehash.hex_to_hash(str(hex_str))
        return phash_obj.hash.flatten()  # (64,) bool array
    except Exception:
        return np.zeros(64, dtype=bool)


def compute_hamming_distances(query_hash: np.ndarray,
                               candidate_hashes: np.ndarray) -> np.ndarray:
    """
    Tính khoảng cách Hamming vectorized giữa 1 query và nhiều candidates.

    Args:
        query_hash      : (64,) bool array — pHash của query
        candidate_hashes: (N, 64) bool array — pHash của N candidates

    Returns:
        distances: (N,) int32 array — khoảng cách Hamming
    """
    return np.sum(query_hash != candidate_hashes, axis=1).astype(np.int32)


def compute_map_at_k(queries_df: pd.DataFrame,
                     gallery_df: pd.DataFrame,
                     top_indices: np.ndarray,
                     k: int = 5) -> float:
    """
    Tính Mean Average Precision tại k (mAP@k) từ kết quả FAISS top-N.

    Args:
        queries_df  : DataFrame query (cột: posting_id, label_group)
        gallery_df  : DataFrame gallery (cột: posting_id, label_group)
        top_indices : (n_queries, N) — chỉ số gallery trả về bởi FAISS
        k           : số kết quả xem xét

    Returns:
        map_score: float — giá trị mAP@k
    """
    gallery_pids    = gallery_df['posting_id'].values
    gallery_labels  = gallery_df['label_group'].values

    # Tiền tính: label_group -> số lượng trong gallery
    label_counts = gallery_df['label_group'].value_counts().to_dict()

    ap_scores = []
    queries_iter = queries_df.reset_index(drop=True)

    for q_idx in range(len(queries_iter)):
        q_pid   = queries_iter.at[q_idx, 'posting_id']
        q_label = queries_iter.at[q_idx, 'label_group']

        # Số relevant items trong gallery (trừ chính query)
        n_relevant = label_counts.get(q_label, 0) - 1
        # Bỏ qua query không có ảnh liên quan trong gallery
        if n_relevant <= 0:
            continue

        # Lấy top-k kết quả, loại bỏ self-match
        retrieved_labels = []
        for gidx in top_indices[q_idx]:
            if gidx < 0:
                break
            if gallery_pids[gidx] != q_pid:
                retrieved_labels.append(gallery_labels[gidx] == q_label)
            if len(retrieved_labels) == k:
                break

        # Tính Average Precision
        hits, precision_sum = 0, 0.0
        for rank, is_relevant in enumerate(retrieved_labels):
            if is_relevant:
                hits += 1
                precision_sum += hits / (rank + 1)

        ap = precision_sum / min(n_relevant, k)
        ap_scores.append(ap)

    return float(np.mean(ap_scores)) if ap_scores else 0.0


def compute_precision_at_1(queries_df: pd.DataFrame,
                            gallery_df: pd.DataFrame,
                            top_indices: np.ndarray) -> float:
    """
    Tính Precision@1: Tỷ lệ query có kết quả đầu tiên đúng nhãn.
    """
    gallery_pids   = gallery_df['posting_id'].values
    gallery_labels = gallery_df['label_group'].values
    queries_iter   = queries_df.reset_index(drop=True)

    correct = 0
    for q_idx in range(len(queries_iter)):
        q_pid   = queries_iter.at[q_idx, 'posting_id']
        q_label = queries_iter.at[q_idx, 'label_group']

        for gidx in top_indices[q_idx]:
            if gidx < 0:
                break
            if gallery_pids[gidx] != q_pid:  # Bỏ self-match
                if gallery_labels[gidx] == q_label:
                    correct += 1
                break

    return correct / len(queries_iter)


def compute_recall_at_k(queries_df: pd.DataFrame,
                         gallery_df: pd.DataFrame,
                         top_indices: np.ndarray,
                         k: int = 5) -> float:
    """
    Tính Recall@k: Tỷ lệ trung bình relevant items được tìm thấy trong top-k.
    """
    gallery_pids   = gallery_df['posting_id'].values
    gallery_labels = gallery_df['label_group'].values
    label_counts   = gallery_df['label_group'].value_counts().to_dict()
    queries_iter   = queries_df.reset_index(drop=True)

    recall_scores = []
    for q_idx in range(len(queries_iter)):
        q_pid      = queries_iter.at[q_idx, 'posting_id']
        q_label    = queries_iter.at[q_idx, 'label_group']
        n_relevant = label_counts.get(q_label, 0) - 1
        if n_relevant <= 0:
            continue

        hits = 0
        count = 0
        for gidx in top_indices[q_idx]:
            if gidx < 0:
                break
            if gallery_pids[gidx] != q_pid:
                if gallery_labels[gidx] == q_label:
                    hits += 1
                count += 1
            if count == k:
                break

        recall = hits / min(n_relevant, k)
        recall_scores.append(recall)

    return float(np.mean(recall_scores)) if recall_scores else 0.0


def build_faiss_index(gallery_features: np.ndarray,
                       use_gpu: bool = False,
                       gpu_resources=None) -> faiss.Index:
    """
    Xây dựng FAISS IndexFlatIP từ gallery features đã L2-normalize.

    Args:
        gallery_features: (N, D) float32 array đã chuẩn hóa L2
        use_gpu         : Có dùng GPU FAISS không
        gpu_resources   : faiss.StandardGpuResources nếu dùng GPU

    Returns:
        index: FAISS index đã thêm gallery
    """
    dim   = gallery_features.shape[1]
    index = faiss.IndexFlatIP(dim)
    if use_gpu and gpu_resources is not None:
        index = faiss.index_cpu_to_gpu(gpu_resources, 0, index)
    index.add(gallery_features.astype(np.float32))
    return index


print('✅ Đã định nghĩa xong tất cả hàm tiện ích!')
print('   - l2_normalize()              : Chuẩn hóa L2 vectorized')
print('   - clean_text()                : Làm sạch văn bản tiêu đề')
print('   - hex_to_phash_array()        : Chuyển pHash hex → bool array')
print('   - compute_hamming_distances() : Khoảng cách Hamming vectorized')
print('   - compute_map_at_k()          : Tính mAP@k từ FAISS results')
print('   - compute_precision_at_1()    : Tính Precision@1')
print('   - compute_recall_at_k()       : Tính Recall@k')
print('   - build_faiss_index()         : Xây FAISS IndexFlatIP')

---
## 📊 Bước 1: Phân chia Validation / Test Set
- **Gallery:** Toàn bộ 34,250 ảnh (không thay đổi)
- **Val Set (20%):** Dùng để tune tham số α và trọng số DINOv2
- **Test Set (80%):** Chỉ dùng một lần duy nhất để báo cáo kết quả cuối
- **Stratify:** Theo `label_group` để đảm bảo phân phối đồng đều

In [ ]:
# ============================================================
# 📊 BƯỚC 1: PHÂN CHIA DỮ LIỆU VALIDATION / TEST
# ============================================================
print('=' * 60)
print('BƯỚC 1: PHÂN CHIA DỮ LIỆU VALIDATION / TEST')
print('=' * 60)

# --- 1.1: Tải file CSV chính ---
print(f'\n📂 Đang tải dữ liệu từ: {CANDIDATE_CSV}')
candidate_df = pd.read_csv(CANDIDATE_CSV)

print(f'✅ Tải xong! Tổng số dòng: {len(candidate_df):,}')
print(f'   Các cột: {list(candidate_df.columns)}')

# Kiểm tra các cột bắt buộc
required_cols = ['posting_id', 'image', 'label_group', 'title']
missing_cols = [c for c in required_cols if c not in candidate_df.columns]
if missing_cols:
    raise ValueError(f'❌ Thiếu cột bắt buộc: {missing_cols}')
print(f'   Kiểm tra cột bắt buộc: ✅ OK')

# Thống kê cơ bản
unique_labels = candidate_df['label_group'].nunique()
avg_per_group = len(candidate_df) / unique_labels
print(f'\n📊 Thống kê dữ liệu:')
print(f'   Tổng số ảnh       : {len(candidate_df):,}')
print(f'   Số label_group    : {unique_labels:,}')
print(f'   TB ảnh/group      : {avg_per_group:.2f}')
print(f'   Max ảnh/group     : {candidate_df["label_group"].value_counts().max()}')
print(f'   Min ảnh/group     : {candidate_df["label_group"].value_counts().min()}')

print('\n   Mẫu dữ liệu (5 dòng đầu):')
display(candidate_df[required_cols + (['image_phash'] if 'image_phash' in candidate_df.columns else [])].head())

# --- 1.2: Phân chia Validation / Test (stratify theo label_group) ---
print('\n✂️  Đang phân chia dữ liệu (stratify theo label_group)...')
print('   - Val Set  : 20% (dùng để tune tham số α, W_DINO)')
print('   - Test Set : 80% (CHỈ DÙNG ĐỂ ĐÁNH GIÁ CUỐI CÙNG)')

val_query_df, test_query_df = train_test_split(
    candidate_df,
    test_size=0.8,
    random_state=42,
    stratify=candidate_df['label_group'].values
)

# Reset index để tránh lỗi khi dùng .at[]
val_query_df  = val_query_df.reset_index(drop=True)
test_query_df = test_query_df.reset_index(drop=True)

# --- 1.3: Lưu kết quả phân chia ---
val_csv_path  = os.path.join(RESULTS_DIR, 'val_query.csv')
test_csv_path = os.path.join(RESULTS_DIR, 'test_query.csv')
val_query_df.to_csv(val_csv_path, index=False)
test_query_df.to_csv(test_csv_path, index=False)

# --- 1.4: In kết quả ---
print(f'\n✅ PHÂN CHIA HOÀN TẤT!')
print(f'   📁 Gallery (toàn bộ)  : {len(candidate_df):,} ảnh  (100%)')
print(f'   📁 Validation Set     : {len(val_query_df):,} ảnh  (20%)  → Lưu tại: val_query.csv')
print(f'   📁 Test Set           : {len(test_query_df):,} ảnh  (80%)  → Lưu tại: test_query.csv')

# Kiểm tra stratification
val_labels  = val_query_df['label_group'].nunique()
test_labels = test_query_df['label_group'].nunique()
print(f'\n📊 Kiểm tra Stratification:')
print(f'   Label groups trong Val Set  : {val_labels:,}')
print(f'   Label groups trong Test Set : {test_labels:,}')
print(f'   Label groups trong Gallery  : {unique_labels:,}')

print(f'\n⚠️  NHẮC NHỞ QUAN TRỌNG:')
print(f'   ✅ Bước 5 (Grid Search α, W_DINO) → CHỈ DÙNG val_query.csv')
print(f'   ✅ Bước 6 (Đánh giá cuối) → CHỈ DÙNG test_query.csv — TUYỆT ĐỐI KHÔNG TUNE!')

---
## 🦉 Bước 2: Trích xuất đặc trưng SigLIP-384 (1152-dim) — Backbone chính
- Load `google/siglip-so400m-patch14-384` từ Hugging Face
- Dùng `SiglipVisionModel` — chỉ image branch
- Resize ảnh lên **384×384** (phiên bản lớn hơn 224, chất lượng cao hơn)
- Output: `pooler_output` 1152 chiều (CLS token)
- Áp dụng L2 Normalization
- Lưu vào `processed_siglip_dino/siglip384_features.npy`


In [ ]:
# ============================================================
# 🦉 BƯỚC 2: TRÍCH XUẤT ĐẶC TRƯNG SigLIP-384 (1152 chiều)
# ============================================================
print('=' * 60)
print('BƯỚC 2: TRÍCH XUẤT ĐẶC TRƯNG SigLIP-384 (BACKBONE CHÍNH)')
print('=' * 60)

# --- 2.1: Custom Dataset cho Shopee (SigLIP) ---
class ShopeeSigLIPDataset(torch.utils.data.Dataset):
    """
    Dataset tùy chỉnh để tải ảnh Shopee cho SigLIP-384.
    Dùng SiglipImageProcessor để tiền xử lý chuẩn.
    """
    def __init__(self, image_dir: str, image_filenames: list, processor):
        self.image_dir       = image_dir
        self.image_filenames = image_filenames
        self.processor       = processor

    def __len__(self) -> int:
        return len(self.image_filenames)

    def __getitem__(self, idx: int):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            # Nếu không mở được ảnh → trả về ảnh xám đặc
            img = Image.new('RGB', (IMG_SIZE_SIGLIP, IMG_SIZE_SIGLIP), (128, 128, 128))

        # SiglipImageProcessor: resize về 384x384, normalize chuẩn
        inputs = self.processor(images=img, return_tensors='pt')
        pixel_values = inputs['pixel_values'].squeeze(0)  # (3, 384, 384)
        return pixel_values, idx


print('✅ Đã tạo ShopeeSigLIPDataset!')


In [ ]:
# --- 2.2: Load mô hình SigLIP-384 từ Hugging Face ---
print(f'🔄 Đang tải mô hình SigLIP ({SIGLIP_MODEL_NAME})...')
print('   (Lần đầu tiên có thể mất 3-5 phút để tải weights ~3GB...)')

siglip_processor = SiglipImageProcessor.from_pretrained(SIGLIP_MODEL_NAME)
siglip_model     = SiglipVisionModel.from_pretrained(SIGLIP_MODEL_NAME)
siglip_model     = siglip_model.to(device)
siglip_model.eval()

# --- 2.3: Xác nhận số chiều output ---
print('\n🔍 Kiểm tra số chiều output SigLIP-384...')
with torch.no_grad():
    dummy_input = torch.randn(2, 3, IMG_SIZE_SIGLIP, IMG_SIZE_SIGLIP).to(device)
    test_output = siglip_model(pixel_values=dummy_input)

siglip_dim = test_output.pooler_output.shape[-1]
print(f'   Output shape   : {test_output.pooler_output.shape}')
print(f'   Feature dim    : {siglip_dim}')

expected_siglip_dim = siglip_model.config.hidden_size
assert siglip_dim == expected_siglip_dim, (
    f'❌ LỖI: Số chiều phải là {expected_siglip_dim}, nhưng nhận được {siglip_dim}!'
)

print(f'\n✅ Mô hình SigLIP-384 đã sẵn sàng!')
print(f'   Kiến trúc    : {SIGLIP_MODEL_NAME}')
print(f'   Feature dim  : {siglip_dim}')
print(f'   Thiết bị     : {next(siglip_model.parameters()).device}')
print(f'   Kiểm tra {expected_siglip_dim}-dim: ✅ PASS')


In [ ]:
# --- 2.4: Hàm trích xuất SigLIP-384 features ---
def extract_siglip_features(image_dir: str,
                             image_filenames: list,
                             model,
                             processor,
                             batch_size: int = 16) -> np.ndarray:
    """
    Trích xuất SigLIP-384 features cho toàn bộ danh sách ảnh.
    Chỉ dùng image branch (không dùng text encoder).

    Args:
        image_dir       : Thư mục chứa ảnh
        image_filenames : Danh sách tên file ảnh
        model           : SiglipVisionModel (đã .eval() và .to(device))
        processor       : SiglipImageProcessor
        batch_size      : Kích thước batch

    Returns:
        features: numpy array shape (N, siglip_dim)
    """
    dataset    = ShopeeSigLIPDataset(image_dir, image_filenames, processor)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False
    )

    all_features = []
    model.eval()

    with torch.no_grad():
        for batch_imgs, _ in tqdm(dataloader, desc='   Trích xuất SigLIP-384', unit='batch'):
            batch_imgs = batch_imgs.to(device, non_blocking=True)

            # Forward pass — chỉ image encoder
            output = model(pixel_values=batch_imgs)

            # pooler_output = CLS token — feature tốt nhất của SigLIP
            img_features = output.pooler_output  # (B, siglip_dim)

            all_features.append(img_features.cpu().float().numpy())

    return np.vstack(all_features).astype(np.float32)


# --- 2.5: Trích xuất (hoặc tải cache nếu đã có) ---
siglip_save_path = os.path.join(PROCESSED_DIR, 'siglip384_features.npy')

if os.path.exists(siglip_save_path):
    print(f'\n📦 Tìm thấy file cache: {siglip_save_path}')
    print('   Đang tải features đã lưu...')
    siglip_features = np.load(siglip_save_path)
    print(f'✅ Đã tải xong! Shape: {siglip_features.shape}')
else:
    print(f'\n🚀 Bắt đầu trích xuất SigLIP-384 features cho {len(candidate_df):,} ảnh...')
    print(f'   Batch size    : {BATCH_SIZE_SIGLIP}')
    print(f'   IMG_SIZE      : {IMG_SIZE_SIGLIP}×{IMG_SIZE_SIGLIP}')
    est_min = len(candidate_df) / BATCH_SIZE_SIGLIP * 1.5 / 60
    print(f'   Ước tính thời gian: {est_min:.1f} phút (T4 GPU)')

    start_time = time.time()

    siglip_features = extract_siglip_features(
        IMAGE_DIR,
        candidate_df['image'].tolist(),
        siglip_model,
        siglip_processor,
        batch_size=BATCH_SIZE_SIGLIP
    )

    elapsed = time.time() - start_time
    print(f'\n✅ Trích xuất hoàn tất!')
    print(f'   Shape     : {siglip_features.shape}')
    print(f'   Thời gian : {elapsed:.1f}s ({elapsed/60:.1f} phút)')
    print(f'   Tốc độ    : {len(candidate_df)/elapsed:.0f} ảnh/giây')

    # Áp dụng L2 Normalization
    siglip_features = l2_normalize(siglip_features)
    print(f'   Đã chuẩn hóa L2 (norm mỗi vector ≈ 1.0)')

    # Lưu features
    np.save(siglip_save_path, siglip_features)
    file_size_mb = os.path.getsize(siglip_save_path) / 1e6
    print(f'\n💾 Đã lưu SigLIP features tại: {siglip_save_path} ({file_size_mb:.1f} MB)')

# Kiểm tra L2-norm từ cache
_norms = np.linalg.norm(siglip_features[:100], axis=1)
if not np.allclose(_norms, 1.0, atol=0.01):
    print('\n🔄 Cache chưa L2-normalize. Đang chuẩn hóa...')
    siglip_features = l2_normalize(siglip_features)
    np.save(siglip_save_path, siglip_features)
    print('✅ Đã chuẩn hóa và lưu lại!')

assert siglip_features.shape == (len(candidate_df), siglip_dim), \
    f'❌ Shape không đúng: {siglip_features.shape}'

print(f'\n📊 Thống kê SigLIP-384 Features:')
print(f'   Shape   : {siglip_features.shape}  ← (N_ảnh, {siglip_dim}_chiều)')
print(f'   Dtype   : {siglip_features.dtype}')
print(f'   Min/Max : [{siglip_features.min():.4f}, {siglip_features.max():.4f}]')
print(f'   Norm TB : {np.linalg.norm(siglip_features[:100], axis=1).mean():.4f} (phải ≈ 1.0)')
print(f'   ✅ Kiểm tra shape và L2-norm: PASS')

# Giải phóng bộ nhớ GPU sau khi trích xuất
del siglip_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n🧹 Đã giải phóng GPU memory của SigLIP!')


---
## 🦕 Bước 3: Trích xuất đặc trưng DINOv2 (768-dim) — Reranker
- Load `dinov2_vitb14` từ PyTorch Hub (facebook research)
- Trích xuất `cls_token` 768 chiều — tốt cho so sánh chi tiết thị giác
- DINOv2 được train hoàn toàn self-supervised → phân biệt tốt texture, pattern
- Lưu vào `processed_siglip_dino/dinov2_features.npy`


In [ ]:
# ============================================================
# 🦕 BƯỚC 3: TRÍCH XUẤT ĐẶC TRƯNG DINOv2 (768-dim RERANKER)
# ============================================================
print('=' * 60)
print('BƯỚC 3: TRÍCH XUẤT ĐẶC TRƯNG DINOv2 (RERANKER)')
print('=' * 60)

# --- 3.1: Custom Dataset cho DINOv2 ---
# DINOv2 dùng ImageNet mean/std chuẩn
dino_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_DINO, IMG_SIZE_DINO),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    ),
])


class ShopeeDINODataset(torch.utils.data.Dataset):
    """
    Dataset tùy chỉnh để tải ảnh Shopee cho DINOv2.
    """
    def __init__(self, image_dir: str, image_filenames: list, transform):
        self.image_dir       = image_dir
        self.image_filenames = image_filenames
        self.transform       = transform

    def __len__(self) -> int:
        return len(self.image_filenames)

    def __getitem__(self, idx: int):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            # Nếu không mở được ảnh → trả về ảnh xám đặc
            img = Image.new('RGB', (IMG_SIZE_DINO, IMG_SIZE_DINO), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, idx


print('✅ Đã tạo ShopeeDINODataset và transform DINOv2!')


In [ ]:
# --- 3.2: Load mô hình DINOv2 từ PyTorch Hub ---
print(f'🔄 Đang tải mô hình DINOv2 ({DINOV2_MODEL_NAME}) từ PyTorch Hub...')
print('   (Lần đầu tiên có thể mất 2-3 phút...)')

# Tải DINOv2 từ facebookresearch qua torch.hub
dino_model = torch.hub.load('facebookresearch/dinov2', DINOV2_MODEL_NAME)
dino_model = dino_model.to(device)
dino_model.eval()

# --- 3.3: Xác nhận số chiều output DINOv2 ---
print('\n🔍 Kiểm tra số chiều output DINOv2...')
with torch.no_grad():
    dummy_dino = torch.randn(2, 3, IMG_SIZE_DINO, IMG_SIZE_DINO).to(device)
    dino_out = dino_model(dummy_dino)

dino_dim = dino_out.shape[-1]
print(f'   Output shape   : {dino_out.shape}')
print(f'   Feature dim    : {dino_dim}')

print(f'\n✅ Mô hình DINOv2 đã sẵn sàng!')
print(f'   Kiến trúc    : {DINOV2_MODEL_NAME}')
print(f'   Feature dim  : {dino_dim}')
print(f'   Thiết bị     : {next(dino_model.parameters()).device}')


In [ ]:
# --- 3.4: Hàm trích xuất DINOv2 features ---
def extract_dinov2_features(image_dir: str,
                             image_filenames: list,
                             model,
                             transform,
                             batch_size: int = 32) -> np.ndarray:
    """
    Trích xuất DINOv2 features (cls_token) cho toàn bộ danh sách ảnh.
    DINOv2 self-forward trả về CLS token — tốt cho instance retrieval.

    Args:
        image_dir       : Thư mục chứa ảnh
        image_filenames : Danh sách tên file ảnh
        model           : DINOv2 model (đã .eval() và .to(device))
        transform       : Transforms tiền xử lý
        batch_size      : Kích thước batch

    Returns:
        features: numpy array shape (N, dino_dim)
    """
    dataset    = ShopeeDINODataset(image_dir, image_filenames, transform)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False
    )

    all_features = []
    model.eval()

    with torch.no_grad():
        for batch_imgs, _ in tqdm(dataloader, desc='   Trích xuất DINOv2', unit='batch'):
            batch_imgs = batch_imgs.to(device, non_blocking=True)

            # DINOv2: forward trả về CLS token (embedding tổng quát tốt nhất)
            features = model(batch_imgs)  # (B, dino_dim)

            all_features.append(features.cpu().float().numpy())

    return np.vstack(all_features).astype(np.float32)


# --- 3.5: Trích xuất (hoặc tải cache nếu đã có) ---
dino_save_path = os.path.join(PROCESSED_DIR, 'dinov2_features.npy')

if os.path.exists(dino_save_path):
    print(f'\n📦 Tìm thấy file cache: {dino_save_path}')
    print('   Đang tải features đã lưu...')
    dino_features = np.load(dino_save_path)
    print(f'✅ Đã tải xong! Shape: {dino_features.shape}')
else:
    print(f'\n🚀 Bắt đầu trích xuất DINOv2 features cho {len(candidate_df):,} ảnh...')
    print(f'   Batch size    : {BATCH_SIZE_DINO}')
    print(f'   IMG_SIZE      : {IMG_SIZE_DINO}×{IMG_SIZE_DINO}')
    est_min = len(candidate_df) / BATCH_SIZE_DINO * 0.5 / 60
    print(f'   Ước tính thời gian: {est_min:.1f} phút (T4 GPU)')

    start_time = time.time()

    dino_features = extract_dinov2_features(
        IMAGE_DIR,
        candidate_df['image'].tolist(),
        dino_model,
        dino_transform,
        batch_size=BATCH_SIZE_DINO
    )

    elapsed = time.time() - start_time
    print(f'\n✅ Trích xuất hoàn tất!')
    print(f'   Shape     : {dino_features.shape}')
    print(f'   Thời gian : {elapsed:.1f}s ({elapsed/60:.1f} phút)')
    print(f'   Tốc độ    : {len(candidate_df)/elapsed:.0f} ảnh/giây')

    # Áp dụng L2 Normalization
    dino_features = l2_normalize(dino_features)
    print(f'   Đã chuẩn hóa L2 (norm mỗi vector ≈ 1.0)')

    # Lưu features
    np.save(dino_save_path, dino_features)
    file_size_mb = os.path.getsize(dino_save_path) / 1e6
    print(f'\n💾 Đã lưu DINOv2 features tại: {dino_save_path} ({file_size_mb:.1f} MB)')

# Kiểm tra L2-norm từ cache
_norms_dino = np.linalg.norm(dino_features[:100], axis=1)
if not np.allclose(_norms_dino, 1.0, atol=0.01):
    print('\n🔄 Cache chưa L2-normalize. Đang chuẩn hóa...')
    dino_features = l2_normalize(dino_features)
    np.save(dino_save_path, dino_features)
    print('✅ Đã chuẩn hóa và lưu lại!')

assert dino_features.shape == (len(candidate_df), dino_dim), \
    f'❌ Shape không đúng: {dino_features.shape}'

print(f'\n📊 Thống kê DINOv2 Features:')
print(f'   Shape   : {dino_features.shape}  ← (N_ảnh, {dino_dim}_chiều)')
print(f'   Dtype   : {dino_features.dtype}')
print(f'   Min/Max : [{dino_features.min():.4f}, {dino_features.max():.4f}]')
print(f'   Norm TB : {np.linalg.norm(dino_features[:100], axis=1).mean():.4f} (phải ≈ 1.0)')
print(f'   ✅ Kiểm tra shape và L2-norm: PASS')

# Giải phóng bộ nhớ GPU
del dino_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n🧹 Đã giải phóng GPU memory của DINOv2!')


---
## 📝 Bước 4: Trích xuất đặc trưng TF-IDF (char_wb + SVD 256-dim)
- Làm sạch văn bản tiêu đề (lowercase, bỏ ký tự đặc biệt)
- `TfidfVectorizer` với `analyzer='char_wb'`, `ngram_range=(3,5)`, `max_features=30000`, `sublinear_tf=True`
- `TruncatedSVD(n_components=256)` để giảm chiều (LSA)
- Áp dụng L2 Normalization
- Lưu vào `processed_siglip_dino/tfidf_features.npy`

In [ ]:
# ============================================================
# 📝 BƯỚC 4: TRÍCH XUẤT ĐẶC TRƯNG TF-IDF + SVD
# ============================================================
print('=' * 60)
print('BƯỚC 4: TRÍCH XUẤT ĐẶC TRƯNG TF-IDF (char_wb, 256-dim SVD)')
print('=' * 60)

# --- 4.1: Làm sạch văn bản tiêu đề ---
print('\n🧹 Đang làm sạch văn bản tiêu đề sản phẩm...')
candidate_df['title_clean'] = candidate_df['title'].apply(clean_text)

title_lengths = candidate_df['title_clean'].str.split().str.len()
print(f'✅ Đã làm sạch {len(candidate_df):,} tiêu đề!')
print(f'   Độ dài TB (từ): {title_lengths.mean():.1f}')
print(f'   Độ dài Max     : {title_lengths.max()}')
print(f'   Độ dài Min     : {title_lengths.min()}')
print(f'\n   Ví dụ làm sạch:')
print(f'   Trước: "{candidate_df["title"].iloc[0]}"')
print(f'   Sau  : "{candidate_df["title_clean"].iloc[0]}"')

# --- 4.2: TF-IDF Vectorizer ---
tfidf_save_path = os.path.join(PROCESSED_DIR, 'tfidf_features.npy')

if os.path.exists(tfidf_save_path):
    print(f'\n📦 Tìm thấy file cache TF-IDF: {tfidf_save_path}')
    tfidf_features = np.load(tfidf_save_path)
    print(f'✅ Đã tải xong! Shape: {tfidf_features.shape}')
    # Kiểm tra và normalize lại nếu cache cũ chưa normalize
    _check_norms = np.linalg.norm(tfidf_features[:100], axis=1)
    if not np.allclose(_check_norms, 1.0, atol=0.05):
        print('   Cache TF-IDF chưa normalize, đang chuẩn hóa...')
        tfidf_features = l2_normalize(tfidf_features)
        np.save(tfidf_save_path, tfidf_features)
        print('   Đã normalize và lưu lại cache!')
else:
    print('\n🔄 Đang chạy TF-IDF Vectorizer (char_wb, ngram 3-5)...')

    # TfidfVectorizer với cấu hình anti-typo tối ưu
    tfidf_vectorizer = TfidfVectorizer(
        analyzer='char_wb',       # Phân tích theo ký tự — kháng lỗi chính tả
        ngram_range=(3, 5),       # Cụm ký tự 3-5 ký tự
        max_features=30000,       # Giới hạn 30,000 features
        sublinear_tf=True,        # Áp dụng 1 + log(tf) thay vì tf thô
        min_df=2,                 # Bỏ terms xuất hiện < 2 lần
        max_df=0.95,              # Bỏ terms xuất hiện > 95% documents
        strip_accents='unicode'   # Chuẩn hóa unicode
    )

    tfidf_sparse = tfidf_vectorizer.fit_transform(candidate_df['title_clean'])
    print(f'✅ TF-IDF Sparse Matrix: {tfidf_sparse.shape}')
    print(f'   Kích thước từ điển : {len(tfidf_vectorizer.vocabulary_):,} char n-grams')
    print(f'   Mật độ sparse      : {tfidf_sparse.nnz / (tfidf_sparse.shape[0] * tfidf_sparse.shape[1]):.6f}')

    # --- 4.3: Giảm chiều bằng TruncatedSVD (LSA) ---
    print(f'\n🔄 Đang giảm chiều bằng TruncatedSVD ({SVD_DIM} chiều)...')
    svd_model = TruncatedSVD(
        n_components=SVD_DIM,
        n_iter=10,
        random_state=42
    )

    tfidf_features = svd_model.fit_transform(tfidf_sparse).astype(np.float32)

    explained_var = svd_model.explained_variance_ratio_.sum()
    print(f'✅ Sau TruncatedSVD: {tfidf_features.shape}')
    print(f'   Tỷ lệ phương sai giải thích: {explained_var:.4f} ({explained_var*100:.2f}%)')

    # L2-normalize
    tfidf_features = l2_normalize(tfidf_features)

    # Lưu features
    np.save(tfidf_save_path, tfidf_features)
    file_size_mb = os.path.getsize(tfidf_save_path) / 1e6
    print(f'\n💾 Đã lưu TF-IDF features tại: {tfidf_save_path} ({file_size_mb:.1f} MB)')

assert tfidf_features.shape == (len(candidate_df), SVD_DIM), \
    f'❌ Shape TF-IDF không đúng: {tfidf_features.shape}'

print(f'\n📊 Thống kê TF-IDF Features:')
print(f'   Shape   : {tfidf_features.shape}  ← (N_ảnh, {SVD_DIM}_chiều_SVD)')
print(f'   Dtype   : {tfidf_features.dtype}')
print(f'   Min/Max : [{tfidf_features.min():.4f}, {tfidf_features.max():.4f}]')
print(f'   ✅ Kiểm tra {SVD_DIM}-dim: PASS')

print(f'\n📦 TÓM TẮT CÁC FEATURES ĐÃ TRÍCH XUẤT:')
print(f'   SigLIP-384 : {siglip_features.shape}  ({siglip_dim}-dim, backbone chính)')
print(f'   DINOv2     : {dino_features.shape}  ({dino_dim}-dim, reranker)')
print(f'   TF-IDF     : {tfidf_features.shape}  ({SVD_DIM}-dim LSA, văn bản)')
print(f'   Fusion     : L2( α×L2(SigLIP) || (1-α)×L2(TF-IDF) )')
print(f'   Rerank     : w_siglip×score_siglip + w_dino×score_dino')


---
## 🔍 Bước 5: Grid Search tham số α trên Validation Set
- Chỉ dùng `val_query.csv` (20% dữ liệu)
- Tìm kiếm α từ 0.30 đến 0.95, bước 0.05
- **Công thức fusion:** `fused = L2( α×L2(SigLIP) || (1-α)×L2(TF-IDF) )`
- Chọn `BEST_ALPHA` có `val_mAP@5` cao nhất

> ⚠️ Grid Search chỉ dùng Validation Set, KHÔNG có DINOv2 reranking ở bước này để tiết kiệm thời gian

In [ ]:
# ============================================================
# 🔍 BƯỚC 5: GRID SEARCH α TRÊN VALIDATION SET
# ============================================================
print('=' * 60)
print('BƯỚC 5: GRID SEARCH α — WEIGHTED LATE FUSION (VAL SET)')
print('=' * 60)

# --- 5.1: Khởi tạo FAISS GPU Resources ---
USE_GPU_FAISS = torch.cuda.is_available()
gpu_resources = None
if USE_GPU_FAISS:
    try:
        gpu_resources = faiss.StandardGpuResources()
        gpu_resources.setTempMemory(512 * 1024 * 1024)  # 512MB temp
        print('✅ FAISS GPU Resources khởi tạo thành công!')
    except Exception as e:
        print(f'⚠️  FAISS GPU thất bại ({e}), dùng CPU thay thế.')
        USE_GPU_FAISS = False
else:
    print('⚠️  Không có GPU — dùng FAISS CPU (chậm hơn).')

# --- 5.2: Tiền tính L2-normalized gallery features ---
print('\n🔄 Đang tiền tính L2-normalized gallery features...')
gallery_img_norm = l2_normalize(siglip_features.astype(np.float32))  # (N, 1152)
gallery_txt_norm = l2_normalize(tfidf_features.astype(np.float32))   # (N, 256)
gallery_dino_norm = l2_normalize(dino_features.astype(np.float32))   # (N, 768)
print(f'✅ Gallery siglip norm : {gallery_img_norm.shape}')
print(f'   Gallery tfidf  norm : {gallery_txt_norm.shape}')
print(f'   Gallery dino   norm : {gallery_dino_norm.shape}')

# --- 5.3: Lấy chỉ số val queries trong gallery ---
print('\n🔄 Đang lấy chỉ số Val Queries trong Gallery...')
posting_id_to_gidx = {
    pid: idx for idx, pid in enumerate(candidate_df['posting_id'])
}

# Đảm bảo tất cả val query đều có trong gallery
missing_pids = [pid for pid in val_query_df['posting_id'] if pid not in posting_id_to_gidx]
if missing_pids:
    print(f'⚠️  {len(missing_pids)} posting_id trong val không có trong gallery!')
else:
    print('   Kiểm tra tính đầy đủ: ✅ OK — tất cả val queries đều có trong gallery')

val_gallery_indices = np.array(
    [posting_id_to_gidx[pid] for pid in val_query_df['posting_id']],
    dtype=np.int64
)

# Trích xuất normalized features của val queries
val_img_norm  = gallery_img_norm[val_gallery_indices]   # (n_val, 1152)
val_txt_norm  = gallery_txt_norm[val_gallery_indices]   # (n_val, 256)
val_dino_norm = gallery_dino_norm[val_gallery_indices]  # (n_val, 768)

print(f'   Số val queries        : {len(val_query_df):,}')
print(f'   Val SigLIP features   : {val_img_norm.shape}')
print(f'   Val TF-IDF features   : {val_txt_norm.shape}')
print(f'   Val DINOv2 features   : {val_dino_norm.shape}')


In [ ]:
# --- 5.4: Vòng lặp Grid Search α ---
alphas = np.arange(0.30, 0.96, 0.05)  # [0.30, 0.35, ..., 0.95]
print(f'🔍 Bắt đầu Grid Search với {len(alphas)} giá trị alpha:')
print(f'   Dải alpha: {[round(a, 2) for a in alphas]}')
print(f'   Công thức: fused = L2( α×L2(SigLIP) || (1-α)×L2(TF-IDF) )')
print(f'   Metric   : mAP@5 trên {len(val_query_df):,} val queries')
print()

best_alpha    = None
best_val_map  = -1.0
grid_results  = []

# Lấy top (TOP_K + buffer) để loại self-match
n_search = TOP_K + 25

start_grid = time.time()

for i, alpha in enumerate(alphas):
    alpha = round(float(alpha), 4)
    beta  = round(1.0 - alpha, 4)

    # ── Xây dựng gallery fusion features ──
    gallery_fused = l2_normalize(
        np.concatenate([alpha * gallery_img_norm, beta * gallery_txt_norm], axis=1)
    )  # (N, 1152+256)

    # ── Xây dựng FAISS IndexFlatIP ──
    faiss_index = build_faiss_index(gallery_fused, USE_GPU_FAISS, gpu_resources)

    # ── Tạo val query fusion features ──
    val_fused = l2_normalize(
        np.concatenate([alpha * val_img_norm, beta * val_txt_norm], axis=1)
    )  # (n_val, 1152+256)

    # ── Tìm kiếm FAISS ──
    _, top_indices = faiss_index.search(val_fused.astype(np.float32), n_search)

    # ── Tính mAP@5 ──
    val_map5 = compute_map_at_k(val_query_df, candidate_df, top_indices, k=TOP_K)

    # Ghi lại kết quả
    grid_results.append({'alpha': alpha, 'beta': beta, 'val_mAP@5': val_map5})

    # Đánh dấu tốt nhất
    is_best = val_map5 > best_val_map
    if is_best:
        best_val_map  = val_map5
        best_alpha    = alpha

    # In kết quả mỗi bước
    marker = ' ◀ TỐT NHẤT!' if is_best else ''
    print(f'   α={alpha:.2f} (SigLIP:{alpha:.0%} | TF-IDF:{beta:.0%})'
          f' → val mAP@5 = {val_map5:.4f}{marker}')

    # Giải phóng bộ nhớ
    del gallery_fused, val_fused, faiss_index
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed_grid = time.time() - start_grid

# --- 5.5: Kết quả Grid Search ---
print()
print('=' * 60)
print(f'✅ GRID SEARCH HOÀN TẤT! (Thời gian: {elapsed_grid:.1f}s)')
print('=' * 60)
print(f'\n   🏆 BEST_ALPHA = {best_alpha:.2f}')
print(f'      SigLIP weight : {best_alpha:.0%}')
print(f'      TF-IDF weight : {(1-best_alpha):.0%}')
print(f'      Val mAP@5     : {best_val_map:.4f}')

grid_df = pd.DataFrame(grid_results)
grid_df['val_mAP@5'] = grid_df['val_mAP@5'].round(4)
grid_df_sorted = grid_df.sort_values('val_mAP@5', ascending=False)

print('\n📊 Bảng kết quả Grid Search (sắp xếp theo val mAP@5):')
display(grid_df_sorted.reset_index(drop=True))

# Lưu kết quả grid search
grid_csv_path = os.path.join(RESULTS_DIR, 'grid_search_siglip_dino.csv')
grid_df.to_csv(grid_csv_path, index=False)
print(f'\n💾 Đã lưu kết quả Grid Search tại: {grid_csv_path}')


In [ ]:
# ============================================================
# 🔍 BƯỚC 5B: GRID SEARCH PHASH_THRESHOLD + W_DINO TRÊN VAL SET
# ============================================================
print('=' * 60)
print('BƯỚC 5B: GRID SEARCH PHASH + W_DINO (VAL SET)')
print('=' * 60)

# Dùng BEST_ALPHA đã tìm được từ Bước 5A
thresholds   = [0, 3, 5, 8, 10]
boost_vals   = [0.1, 0.15, 0.2, 0.3]
w_dino_vals  = [0.3, 0.4, 0.5, 0.6, 0.7]  # Trọng số DINOv2 trong rerank

best_threshold = PHASH_THRESHOLD
best_boost     = PHASH_BOOST
best_w_dino    = W_DINO
best_val_map_b = -1.0

HAS_PHASH_COL_5B = 'image_phash' in candidate_df.columns

if HAS_PHASH_COL_5B:
    # Tiền tính pHash arrays
    phash_cache_path_5b = os.path.join(PROCESSED_DIR, 'phash_arrays.npy')
    if os.path.exists(phash_cache_path_5b):
        gallery_phash_arrays_5b = np.load(phash_cache_path_5b)
        print(f'✅ Tải pHash cache! Shape: {gallery_phash_arrays_5b.shape}')
    else:
        phash_list_5b = [
            hex_to_phash_array(h)
            for h in tqdm(candidate_df['image_phash'], desc='   Chuyển pHash')
        ]
        gallery_phash_arrays_5b = np.array(phash_list_5b, dtype=bool)
        np.save(phash_cache_path_5b, gallery_phash_arrays_5b)
        print(f'✅ Tạo pHash arrays! Shape: {gallery_phash_arrays_5b.shape}')

    # Xây FAISS index một lần với best_alpha
    best_beta_5b = round(1.0 - best_alpha, 4)
    gallery_fused_5b = l2_normalize(
        np.concatenate([
            best_alpha   * gallery_img_norm,
            best_beta_5b * gallery_txt_norm
        ], axis=1)
    )
    val_index_5b = build_faiss_index(gallery_fused_5b, USE_GPU_FAISS, gpu_resources)

    # Val query fusion features
    val_fused_5b = l2_normalize(
        np.concatenate([
            best_alpha   * gallery_img_norm[val_gallery_indices],
            best_beta_5b * gallery_txt_norm[val_gallery_indices]
        ], axis=1)
    )

    # FAISS search một lần — dùng lại cho mọi tổ hợp tham số
    val_top_scores_5b, val_top_raw_5b = val_index_5b.search(
        val_fused_5b.astype(np.float32), TOP_RERANK + 5
    )

    gallery_pids_5b = candidate_df['posting_id'].values
    n_val_5b = len(val_query_df)
    val_query_reset_5b = val_query_df.reset_index(drop=True)

    print(f'\nĐang thử {len(w_dino_vals)} W_DINO x {len(thresholds)} threshold x {len(boost_vals)} boost...')
    print(f'   BEST_ALPHA = {best_alpha:.2f}  |  TOP_RERANK = {TOP_RERANK}')
    print()

    for w_dino in w_dino_vals:
        w_siglip = round(1.0 - w_dino, 4)
        for thresh in thresholds:
            for boost in boost_vals:
                val_reranked_5b = np.full((n_val_5b, TOP_K), -1, dtype=np.int64)

                for q_idx in range(n_val_5b):
                    q_pid = val_query_reset_5b.at[q_idx, 'posting_id']
                    cands_gidx, cands_siglip_scores = [], []

                    # Thu thập top-RERANK ứng viên (loại self-match)
                    for gidx, score in zip(val_top_raw_5b[q_idx], val_top_scores_5b[q_idx]):
                        if gidx < 0: break
                        if gallery_pids_5b[gidx] == q_pid: continue
                        cands_gidx.append(gidx)
                        cands_siglip_scores.append(float(score))
                        if len(cands_gidx) == TOP_RERANK: break

                    if not cands_gidx: continue
                    cands_gidx          = np.array(cands_gidx, dtype=np.int64)
                    cands_siglip_scores = np.array(cands_siglip_scores, dtype=np.float32)

                    # DINOv2 reranking: tính cosine similarity DINOv2
                    q_gidx_val    = val_gallery_indices[q_idx]
                    q_dino_vec    = gallery_dino_norm[q_gidx_val]       # (768,)
                    c_dino_vecs   = gallery_dino_norm[cands_gidx]       # (n_cands, 768)
                    dino_scores   = c_dino_vecs @ q_dino_vec            # (n_cands,) cosine sim

                    # Kết hợp điểm: w_siglip * siglip + w_dino * dino
                    combined_scores = w_siglip * cands_siglip_scores + w_dino * dino_scores

                    # pHash Boost
                    q_phash = gallery_phash_arrays_5b[q_gidx_val]
                    c_phashes = gallery_phash_arrays_5b[cands_gidx]
                    ham = compute_hamming_distances(q_phash, c_phashes)
                    combined_scores[ham <= thresh] += boost

                    order = np.argsort(-combined_scores)[:TOP_K]
                    val_reranked_5b[q_idx] = cands_gidx[order]

                val_map_5b = compute_map_at_k(val_query_df, candidate_df, val_reranked_5b, k=TOP_K)

                is_best_5b = val_map_5b > best_val_map_b
                if is_best_5b:
                    best_val_map_b = val_map_5b
                    best_threshold = thresh
                    best_boost     = boost
                    best_w_dino    = w_dino

                marker_5b = ' ◄ TỐT NHẤT!' if is_best_5b else ''
                print(f'   w_dino={w_dino:.1f}, thresh={thresh:2d}, boost={boost:.2f} '
                      f'→ val mAP@5={val_map_5b:.4f}{marker_5b}')

    print(f'\n🏆 BEST params:')
    print(f'   W_DINO         = {best_w_dino}  (W_SIGLIP = {1-best_w_dino})')
    print(f'   pHash thresh   = {best_threshold}')
    print(f'   pHash boost    = {best_boost}')
    print(f'   val mAP@5      = {best_val_map_b:.4f}')

    # Cập nhật tham số tối ưu
    PHASH_THRESHOLD = best_threshold
    PHASH_BOOST     = best_boost
    W_DINO          = best_w_dino
    W_SIGLIP        = round(1.0 - W_DINO, 4)
    print(f'\n✅ Đã cập nhật: PHASH_THRESHOLD={PHASH_THRESHOLD}, PHASH_BOOST={PHASH_BOOST}')
    print(f'   W_DINO={W_DINO}, W_SIGLIP={W_SIGLIP}')

    # Dọn bộ nhớ
    del gallery_fused_5b, val_fused_5b, val_index_5b
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    print('⚠️  Không có cột image_phash → bỏ qua grid search threshold')
    print(f'   Giữ mặc định: PHASH_THRESHOLD={PHASH_THRESHOLD}, W_DINO={W_DINO}')
    gallery_phash_arrays_5b = None


---
## 🏆 Bước 6: Đánh giá cuối cùng trên Test Set (Một lần duy nhất!)
- Áp dụng `BEST_ALPHA` từ Bước 5
- **AQE:** Mở rộng query bằng trung bình top-3 gallery vectors
- **DINOv2 Reranking:** Tính lại điểm bằng DINOv2 cosine similarity cho top-100
- **pHash Boost:** Cộng điểm cho ảnh có pHash distance ≤ threshold
- **Final Rerank:** Điểm tổng hợp = W_SIGLIP×siglip + W_DINO×dino + pHash_boost
- Tính `mAP@5`, `Precision@1`, `Recall@5` trên **Test Set**

> ⚠️ **CẢNH BÁO:** Ô này chỉ được chạy **MỘT LẦN DUY NHẤT** sau khi đã cố định tất cả tham số từ Bước 5!

In [ ]:
# ============================================================
# 🏆 BƯỚC 6: ĐÁNH GIÁ CUỐI CÙNG TRÊN TEST SET
# ============================================================
print('=' * 60)
print('BƯỚC 6: ĐÁNH GIÁ CUỐI CÙNG — TEST SET + DINO RERANK + pHASH')
print('=' * 60)

print(f'\n⚙️  Tham số sử dụng:')
print(f'   BEST_ALPHA      = {best_alpha:.2f}  (SigLIP:{best_alpha:.0%} | TF-IDF:{(1-best_alpha):.0%})')
print(f'   W_SIGLIP        = {W_SIGLIP:.2f}  (trọng số SigLIP trong rerank)')
print(f'   W_DINO          = {W_DINO:.2f}  (trọng số DINOv2 trong rerank)')
print(f'   PHASH_THRESHOLD = {PHASH_THRESHOLD}  (khoảng cách Hamming tối đa để boost)')
print(f'   PHASH_BOOST     = {PHASH_BOOST}  (giá trị cộng thêm điểm số)')
print(f'   TOP_RERANK      = {TOP_RERANK}  (số ứng viên trước khi rerank)')
print(f'   AQE_K           = {AQE_K}  (số top results để mở rộng query)')
print(f'   TOP_K           = {TOP_K}  (số kết quả cuối cùng)')

# --- 6.1: Chuẩn bị pHash arrays ---
HAS_PHASH_COL = 'image_phash' in candidate_df.columns

if HAS_PHASH_COL:
    print(f'\n🔄 Đang tiền tính pHash arrays cho {len(candidate_df):,} ảnh...')
    phash_cache_path = os.path.join(PROCESSED_DIR, 'phash_arrays.npy')

    if os.path.exists(phash_cache_path):
        gallery_phash_arrays = np.load(phash_cache_path)
        print(f'✅ Đã tải pHash cache! Shape: {gallery_phash_arrays.shape}')
    else:
        phash_list = [
            hex_to_phash_array(h)
            for h in tqdm(candidate_df['image_phash'], desc='   Chuyển pHash')
        ]
        gallery_phash_arrays = np.array(phash_list, dtype=bool)
        np.save(phash_cache_path, gallery_phash_arrays)
        print(f'✅ Đã tạo và lưu pHash arrays! Shape: {gallery_phash_arrays.shape}')
else:
    print('\n⚠️  Không tìm thấy cột "image_phash" trong CSV.')
    print('   Bỏ qua pHash Boosting — chỉ dùng FAISS + DINOv2 scores.')
    gallery_phash_arrays = None


In [ ]:
# --- 6.2: Xây dựng Gallery FAISS Index với BEST_ALPHA ---
print(f'\n🔄 Xây dựng FAISS Index với BEST_ALPHA={best_alpha:.2f}...')

best_beta = round(1.0 - best_alpha, 4)

# Tính gallery fused features: L2( α×SigLIP || (1-α)×TF-IDF )
final_gallery_fused = l2_normalize(
    np.concatenate([
        best_alpha * gallery_img_norm,
        best_beta  * gallery_txt_norm
    ], axis=1)
)  # (N, siglip_dim+SVD_DIM)

print(f'   Gallery fused shape : {final_gallery_fused.shape}')
print(f'   Chiều fusion total  : {siglip_dim}×{best_alpha} + {SVD_DIM}×{best_beta} → {siglip_dim+SVD_DIM} → L2')

# Xây dựng FAISS IndexFlatIP
final_index = build_faiss_index(final_gallery_fused, USE_GPU_FAISS, gpu_resources)
print(f'   FAISS Index type    : IndexFlatIP (Inner Product = Cosine sau L2)')
print(f'   FAISS trên GPU      : {USE_GPU_FAISS}')
print(f'✅ Đã xây dựng FAISS Index với {final_index.ntotal:,} gallery items!')

# --- 6.3: Chuẩn bị Test Query Features ---
print(f'\n🔄 Chuẩn bị Test Query Features...')
test_gallery_indices = np.array(
    [posting_id_to_gidx[pid] for pid in test_query_df['posting_id']],
    dtype=np.int64
)

test_fused = l2_normalize(
    np.concatenate([
        best_alpha * gallery_img_norm[test_gallery_indices],
        best_beta  * gallery_txt_norm[test_gallery_indices]
    ], axis=1)
)  # (n_test, siglip_dim+SVD_DIM)

print(f'   Số test queries     : {len(test_query_df):,}')
print(f'   Test fused shape    : {test_fused.shape}')

# --- 6.4: FAISS Search (top TOP_RERANK để rerank) ---
print(f'\n🔍 Đang tìm kiếm top-{TOP_RERANK} ứng viên cho {len(test_query_df):,} test queries...')
start_search = time.time()

top_scores_raw, top_indices_raw = final_index.search(
    test_fused.astype(np.float32),
    TOP_RERANK + 5  # Buffer thêm 5 để loại self-match
)

elapsed_search = time.time() - start_search
print(f'✅ Tìm kiếm hoàn tất! Thời gian: {elapsed_search:.2f}s')
print(f'   Tốc độ: {len(test_query_df)/elapsed_search:.0f} queries/giây')


In [ ]:
# ============================================================
# BƯỚC 6.4b: AVERAGE QUERY EXPANSION (AQE)
# ============================================================
# AQE: Mở rộng query bằng cách lấy trung bình query + top-AQE_K results
# Giúp tăng recall đáng kể mà không cần thêm dữ liệu
print('=' * 60)
print('BƯỚC 6.4b: AVERAGE QUERY EXPANSION (AQE)')
print('=' * 60)

DO_AQE = True  # Đặt False để so sánh không có AQE

if DO_AQE:
    print(f'\n[AQE] Đang áp dụng AQE với top-{AQE_K} results...')
    print(f'   AQE công thức: q_expanded = L2_Norm(q + mean(top_{AQE_K}_gallery_vecs))')

    gallery_pids_aqe = candidate_df['posting_id'].values
    n_test_aqe = len(test_query_df)
    test_fused_expanded = test_fused.copy()  # (n_test, fused_dim)

    expand_count = 0
    qreset = test_query_df.reset_index(drop=True)

    for q_idx in range(n_test_aqe):
        q_pid = qreset.at[q_idx, 'posting_id']

        # Thu thập top-AQE_K valid results (bỏ self-match)
        neighbors = []
        for gidx in top_indices_raw[q_idx]:
            if gidx < 0:
                break
            if gallery_pids_aqe[gidx] == q_pid:
                continue
            neighbors.append(gidx)
            if len(neighbors) == AQE_K:
                break

        if len(neighbors) == 0:
            continue

        # Lấy fused vectors của neighbors từ gallery
        neighbor_vecs = final_gallery_fused[neighbors]  # (AQE_K, fused_dim)

        # AQE: query mới = L2_Norm(query + mean(neighbors))
        q_expanded = test_fused[q_idx] + neighbor_vecs.mean(axis=0)
        q_norm = np.linalg.norm(q_expanded)
        if q_norm > 1e-10:
            test_fused_expanded[q_idx] = (q_expanded / q_norm).astype(np.float32)
        expand_count += 1

    print(f'[AQE] Hoàn tất! Đã mở rộng {expand_count:,}/{n_test_aqe:,} queries')

    # Search lại với expanded queries
    print(f'\n[AQE] Đang search lại với expanded queries...')
    top_scores_raw, top_indices_raw = final_index.search(
        test_fused_expanded.astype(np.float32),
        TOP_RERANK + 5
    )
    print(f'[AQE] Search hoàn tất! top_indices_raw đã được cập nhật.')
else:
    print('[AQE] AQE bị tắt (DO_AQE=False) — dùng kết quả search gốc')


In [ ]:
# --- 6.5: DINOv2 Reranking + pHash Boosting ---
print(f'\n🔄 Đang áp dụng DINOv2 Reranking + pHash Boost...')
print(f'   W_SIGLIP  = {W_SIGLIP:.2f}  |  W_DINO = {W_DINO:.2f}')
print(f'   pHash Threshold : <= {PHASH_THRESHOLD}  |  Boost = +{PHASH_BOOST}')
print(f'   Số ứng viên     : top-{TOP_RERANK} (trước rerank)')
print(f'   Kết quả cuối    : top-{TOP_K}')

gallery_pids   = candidate_df['posting_id'].values
n_test         = len(test_query_df)
queries_reset  = test_query_df.reset_index(drop=True)

# Ma trận lưu kết quả cuối: (n_test, TOP_K) gallery indices
final_top_indices = np.full((n_test, TOP_K), fill_value=-1, dtype=np.int64)

for q_idx in tqdm(range(n_test), desc='   Reranking', unit='query'):
    q_pid  = queries_reset.at[q_idx, 'posting_id']

    # ── Bước A: Lấy top-RERANK ứng viên, loại self-match ──
    candidates_gidx          = []
    candidates_siglip_scores = []

    for gidx, score in zip(top_indices_raw[q_idx], top_scores_raw[q_idx]):
        if gidx < 0:
            break
        if gallery_pids[gidx] == q_pid:
            continue  # Loại self-match
        candidates_gidx.append(gidx)
        candidates_siglip_scores.append(float(score))
        if len(candidates_gidx) == TOP_RERANK:
            break

    if not candidates_gidx:
        continue

    candidates_gidx          = np.array(candidates_gidx, dtype=np.int64)
    candidates_siglip_scores = np.array(candidates_siglip_scores, dtype=np.float32)

    # ── Bước B: DINOv2 Reranking — tính cosine similarity thuần thị giác ──
    q_gidx      = test_gallery_indices[q_idx]
    q_dino_vec  = gallery_dino_norm[q_gidx]              # (768,)
    c_dino_vecs = gallery_dino_norm[candidates_gidx]     # (n_cands, 768)
    dino_scores = c_dino_vecs @ q_dino_vec               # (n_cands,) cosine sim

    # ── Kết hợp điểm: W_SIGLIP * siglip_score + W_DINO * dino_score ──
    combined_scores = W_SIGLIP * candidates_siglip_scores + W_DINO * dino_scores

    # ── Bước C: pHash Boosting (nếu có pHash) ──
    if gallery_phash_arrays is not None:
        q_phash   = gallery_phash_arrays[q_gidx]              # (64,) bool
        c_phashes = gallery_phash_arrays[candidates_gidx]     # (n_cands, 64) bool

        # Tính khoảng cách Hamming vectorized
        ham_dists = compute_hamming_distances(q_phash, c_phashes)  # (n_cands,)

        # Boost: cộng PHASH_BOOST vào candidates có khoảng cách <= threshold
        boost_mask = ham_dists <= PHASH_THRESHOLD
        combined_scores[boost_mask] += PHASH_BOOST

    # ── Bước D: Rerank theo điểm tổng hợp, lấy top-K ──
    rerank_order = np.argsort(-combined_scores)[:TOP_K]
    final_top_indices[q_idx] = candidates_gidx[rerank_order]

print(f'✅ Reranking hoàn tất!')


In [ ]:
# --- 6.6: Tính Final Metrics trên Test Set ---
print(f'\n📊 Đang tính Final Metrics trên Test Set...')
print('   (Đây là kết quả CHÍNH THỨC, chỉ chạy 1 lần)')

final_map5    = compute_map_at_k(test_query_df, candidate_df, final_top_indices, k=5)
final_prec1   = compute_precision_at_1(test_query_df, candidate_df, final_top_indices)
final_recall5 = compute_recall_at_k(test_query_df, candidate_df, final_top_indices, k=5)

# --- 6.7: So sánh với baseline ---
BASELINE_MAP5 = 0.7635
TARGET_MAP5   = 0.80

improvement = final_map5 - BASELINE_MAP5
target_met  = final_map5 >= TARGET_MAP5

print()
print('╔═══════════════════════════════════════════════════════════╗')
print('║    🏆 KẾT QUẢ CUỐI CÙNG — TUẦN 4 (SigLIP-384 + DINOv2) ║')
print('╠═══════════════════════════════════════════════════════════╣')
print(f'║  Test Set           : {len(test_query_df):,} queries (80%)               ║')
print(f'║  BEST_ALPHA         : {best_alpha:.2f} (SigLIP:{best_alpha:.0%} | TF-IDF:{best_beta:.0%})    ║')
print(f'║  Rerank Weights     : W_SIGLIP={W_SIGLIP:.2f} | W_DINO={W_DINO:.2f}      ║')
print(f'║  pHash Boost        : Threshold={PHASH_THRESHOLD}, Boost={PHASH_BOOST}               ║')
print(f'║  AQE                : K={AQE_K} (Average Query Expansion)          ║')
print('╠═══════════════════════════════════════════════════════════╣')
print(f'║  mAP@5              : {final_map5:.4f}                               ║')
print(f'║  Precision@1        : {final_prec1:.4f}                               ║')
print(f'║  Recall@5           : {final_recall5:.4f}                               ║')
print('╠═══════════════════════════════════════════════════════════╣')
print(f'║  Baseline (T3)      : {BASELINE_MAP5:.4f}  (ResNet50)                 ║')
print(f'║  Cải thiện          : {improvement:+.4f}  ({improvement/BASELINE_MAP5*100:+.2f}%)                  ║')
print(f'║  Mục tiêu >= 0.80   : {"✅ ĐẠT!" if target_met else "❌ CHƯA ĐẠT"}                              ║')
print('╚═══════════════════════════════════════════════════════════╝')

if target_met:
    print(f'\n🎉 CHÚC MỪNG! Đã đạt mục tiêu mAP@5 >= 0.80!')
    print(f'   Cải thiện {improvement:.4f} điểm ({improvement/BASELINE_MAP5*100:.2f}%) so với baseline ResNet50!')
    print(f'   Chiến lược SigLIP-384 + DINOv2 Reranking + AQE + pHash đã hiệu quả!')
else:
    gap = TARGET_MAP5 - final_map5
    print(f'\n⚠️  Chưa đạt mục tiêu. Còn cách {gap:.4f} điểm.')
    print('   Gợi ý thêm:')
    print('     - Thử SigLIP Large (google/siglip-large-patch16-384)')
    print('     - Tăng TOP_RERANK lên 200')
    print('     - Thử DINOv2 vitl14 thay vì vitb14 (1024-dim)')
    print('     - Thêm Color Histogram làm đặc trưng thứ 4')


In [ ]:
# --- 6.8: Lưu Final Metrics ra CSV ---
final_metrics = {
    'metric'         : ['mAP@5', 'Precision@1', 'Recall@5'],
    'value'          : [final_map5, final_prec1, final_recall5],
    'baseline_T3'    : [BASELINE_MAP5, None, None],
    'improvement'    : [final_map5 - BASELINE_MAP5, None, None],
    'target'         : [TARGET_MAP5, None, None],
    'target_met'     : [target_met, None, None],
}

# Thêm metadata đầy đủ
metadata = {
    'metric'         : ['--METADATA--', 'best_alpha', 'w_siglip', 'w_dino',
                        'phash_threshold', 'phash_boost', 'aqe_k',
                        'top_rerank', 'top_k', 'svd_dim',
                        'model_backbone', 'model_reranker', 'model_text',
                        'n_gallery', 'n_val_queries', 'n_test_queries'],
    'value'          : [None, best_alpha, W_SIGLIP, W_DINO,
                        PHASH_THRESHOLD, PHASH_BOOST, AQE_K,
                        TOP_RERANK, TOP_K, SVD_DIM,
                        SIGLIP_MODEL_NAME, DINOV2_MODEL_NAME, 'TF-IDF(char_wb)+SVD(256)',
                        len(candidate_df), len(val_query_df), len(test_query_df)],
    'baseline_T3'    : [None] * 16,
    'improvement'    : [None] * 16,
    'target'         : [None] * 16,
    'target_met'     : [None] * 16,
}

metrics_df = pd.concat([
    pd.DataFrame(final_metrics),
    pd.DataFrame(metadata)
], ignore_index=True)

metrics_csv_path = os.path.join(RESULTS_DIR, 'final_metrics_siglip_dino.csv')
metrics_df.to_csv(metrics_csv_path, index=False)

print(f'💾 Đã lưu kết quả cuối tại: {metrics_csv_path}')
print()
display(metrics_df.head(3))

# --- 6.9: Tóm tắt tất cả các file đã tạo ---
print('\n📁 DANH SÁCH FILE ĐÃ TẠO:')
output_files = [
    (os.path.join(PROCESSED_DIR, 'siglip384_features.npy'),   f'SigLIP-384 gallery features (N×{siglip_dim})'),
    (os.path.join(PROCESSED_DIR, 'dinov2_features.npy'),      f'DINOv2 gallery features (N×{dino_dim})'),
    (os.path.join(PROCESSED_DIR, 'tfidf_features.npy'),       f'TF-IDF gallery features (N×{SVD_DIM})'),
    (os.path.join(PROCESSED_DIR, 'phash_arrays.npy'),         'pHash bool arrays (N×64)'),
    (os.path.join(RESULTS_DIR,   'val_query.csv'),             'Validation query set (20%)'),
    (os.path.join(RESULTS_DIR,   'test_query.csv'),            'Test query set (80%)'),
    (os.path.join(RESULTS_DIR,   'grid_search_siglip_dino.csv'), 'Grid Search results (α sweep)'),
    (os.path.join(RESULTS_DIR,   'final_metrics_siglip_dino.csv'), 'Final evaluation metrics'),
]

for fpath, fdesc in output_files:
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) / 1e6 if exists else 0
    status = f'✅ {size:.1f} MB' if exists else '❌ Chưa tạo'
    print(f'   {status:15s}  {fdesc:50s}  {os.path.basename(fpath)}')


---
## 📝 Tổng kết — Tuần 4 (SigLIP-384 + DINOv2)

### 🔧 Những gì đã thực hiện:

| Bước | Nội dung | Chi tiết |
|------|----------|-----------|
| **1** | Phân chia dữ liệu | `train_test_split(test_size=0.8, stratify=label_group, random_state=42)` |
| **2** | SigLIP-384 Features | `SiglipVisionModel(google/siglip-so400m-patch14-384)` → pooler_output (1152-dim) → L2-norm |
| **3** | DINOv2 Features | `torch.hub(facebookresearch/dinov2, dinov2_vitb14)` → cls_token (768-dim) → L2-norm |
| **4** | TF-IDF Features | `TfidfVectorizer(char_wb, (3,5), 30k)` → `TruncatedSVD(256)` → L2-norm |
| **5** | Grid Search | α ∈ [0.30, 0.95] step 0.05 → BEST_ALPHA. W_DINO grid search trên Val Set |
| **6** | Đánh giá cuối | AQE + DINOv2 Rerank + pHash Boost → Final mAP@5, P@1, R@5 trên Test Set |

### 🧠 Tại sao SigLIP-384 + DINOv2 hiệu quả?
- **SigLIP-384:** Trained contrastive với text → hiểu ngữ nghĩa toàn cục ("áo thun nam màu đỏ")
- **DINOv2:** Self-supervised learning → phân biệt chi tiết thị giác (vải, họa tiết, kết cấu)
- **Kết hợp 2 mô hình:** "Best of both worlds" — SigLIP tìm đúng danh mục, DINOv2 rerank đúng sản phẩm

### 📊 Kết quả (điền vào sau khi chạy):

| Metric | Tuần 3 (Baseline) | Tuần 4 (SigLIP+DINOv2) | Cải thiện |
|--------|-------------------|------------------------|----------|
| **mAP@5** | 0.7635 | *(xem ô trên)* | *(xem ô trên)* |
| **Precision@1** | N/A | *(xem ô trên)* | — |
| **Recall@5** | N/A | *(xem ô trên)* | — |

### 📌 Ghi chú cho Tuần 5 (nếu chưa đạt mục tiêu):
- Thử `google/siglip-large-patch16-384` (1152-dim, ViT-L) thay vì so400m
- Thử `dinov2_vitl14` (1024-dim) thay vì `dinov2_vitb14`
- Tăng `TOP_RERANK` lên 200 để rerank kỹ hơn
- Thêm **Color Histogram** như thành phần thứ 4 trong fusion
- Thử **DBA (Database-side Augmentation)** thay vì AQE

---
*Notebook: `Tuan4_GiaVy_SigLIP_DINOv2.ipynb` | Tác giả: Mã Gia Vỹ | Nhóm 3*
